In [1]:
import pandas as pd
import time

def scrape_stat_category(stat_id):
    base_url = f"https://www.ncaa.com/stats/lacrosse-women/d1/current/team/{stat_id}"
    all_pages = []
    page = 1

    while True:
        url = base_url if page == 1 else f"{base_url}/p{page}"
        print(f"Scraping: {url}")

        try:
            tables = pd.read_html(url)
            df = tables[0]

            if df.shape[0] == 0:
                break

            all_pages.append(df)
            page += 1
            time.sleep(2)

        except:
            break

    return pd.concat(all_pages, ignore_index=True)

In [2]:
def build_master_dataset():

    stat_categories = {
        "assists": 480,
        "draw_controls": 262,
        "free_position": 1082,
        "shots_on_goal": 1162,
        "shots_per_game": 1160,
        "goals": 246,
        "caused_turnovers": 264
    }

    all_stats = {}
    games_df = None

    for stat_name, stat_id in stat_categories.items():
        df = scrape_stat_category(stat_id)

        df.columns = df.columns.str.lower().str.replace(" ", "_")
        df = df[df["team"] != "Team"]
        df = df.dropna(subset=["team"])

        if "games" in df.columns and games_df is None:
            games_df = df[["team", "games"]].copy()
            games_df = games_df.rename(columns={"games": "games_played"})

        df = df.drop(columns=["rank", "games"], errors="ignore")
        stat_column = df.columns[-1]
        df = df[["team", stat_column]]
        df = df.rename(columns={stat_column: stat_name})

        all_stats[stat_name] = df

    master_df = games_df

    for df in all_stats.values():
        master_df = master_df.merge(df, on="team", how="outer")

    for col in master_df.columns:
        if col != "team":
            master_df[col] = pd.to_numeric(master_df[col], errors="coerce")

    return master_df

In [7]:
def add_offensive_metrics(df):

    df = df.copy()

    df["Total_Shots"] = df["shots_per_game"] * df["games_played"]

    df["Estimated_Possessions"] = df["Total_Shots"] + df["caused_turnovers"]

    df = df.replace([float("inf"), -float("inf")], None)

    df["Offensive_Efficiency"] = df["goals"] / df["Estimated_Possessions"]
    df["Shot_Efficiency"] = df["goals"] / df["Total_Shots"]
    df["Turnover_Rate"] = df["caused_turnovers"] / df["Estimated_Possessions"]
    df["Pace"] = df["Estimated_Possessions"] / df["games_played"]

    return df.dropna()

In [8]:
import plotly.express as px
import plotly.graph_objects as go

In [9]:
def build_kpi_panel(df):

    top_team = df.sort_values("Offensive_Efficiency", ascending=False).iloc[0]

    print("Most Efficient Offense:", top_team["team"])
    print("Offensive Efficiency:", round(top_team["Offensive_Efficiency"], 3))
    print("League Avg Efficiency:", round(df["Offensive_Efficiency"].mean(), 3))
    print("Fastest Pace:", df.sort_values("Pace", ascending=False).iloc[0]["team"])
    print("Lowest Turnover Rate:", df.sort_values("Turnover_Rate").iloc[0]["team"])

In [10]:
def plot_efficiency_ranking(df):

    df_sorted = df.sort_values("Offensive_Efficiency", ascending=False)

    fig = px.bar(
        df_sorted,
        x="Offensive_Efficiency",
        y="team",
        orientation="h",
        title="Offensive Efficiency Ranking",
        color="Offensive_Efficiency",
        color_continuous_scale="Reds"
    )

    fig.update_layout(
        template="plotly_dark",
        yaxis=dict(autorange="reversed"),
        height=900
    )

    fig.show()

In [11]:
def plot_efficiency_vs_pace(df):

    fig = px.scatter(
        df,
        x="Pace",
        y="Offensive_Efficiency",
        size="Estimated_Possessions",
        color="Offensive_Efficiency",
        hover_name="team",
        color_continuous_scale="Reds",
        title="Offensive Efficiency vs Pace"
    )

    fig.update_layout(template="plotly_dark")

    fig.show()

In [12]:
def plot_shot_efficiency(df):

    df_sorted = df.sort_values("Shot_Efficiency", ascending=False)

    fig = px.bar(
        df_sorted,
        x="Shot_Efficiency",
        y="team",
        orientation="h",
        title="Shot Efficiency Ranking",
        color="Shot_Efficiency",
        color_continuous_scale="Reds"
    )

    fig.update_layout(
        template="plotly_dark",
        yaxis=dict(autorange="reversed"),
        height=900
    )

    fig.show()

In [13]:
master_df = build_master_dataset()
master_df = add_offensive_metrics(master_df)

build_kpi_panel(master_df)
plot_efficiency_ranking(master_df)
plot_efficiency_vs_pace(master_df)
plot_shot_efficiency(master_df)

Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/480
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/480/p2
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/480/p3
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/480/p4
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/262
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/262/p2
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/262/p3
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/262/p4
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/1082
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/1082/p2
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/1082/p3
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/1082/p4
Scraping: https://www.ncaa.com/stats/lacrosse-women/d1/current/team/1162
Scraping: https://www.ncaa.com/s